# v19 SFT Training Policy

Self-contained Kaggle/Colab notebook for the v19 supervised warm-start policy. Configure `HF_TOKEN` as a Kaggle Secret or environment variable before running.

In [ ]:
import os


def _read_kaggle_secret(names):
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception as exc:
        print(f'Kaggle Secrets unavailable: {exc.__class__.__name__}')
        return ''

    client = UserSecretsClient()
    for name in names:
        try:
            value = (client.get_secret(name) or '').strip()
        except Exception as exc:
            print(f'Kaggle Secret {name} unavailable: {exc.__class__.__name__}')
            value = ''
        if value:
            return value
    return ''


def _resolve_hf_token():
    token = (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN') or '').strip()
    if not token:
        token = _read_kaggle_secret(('HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'))
    if not token:
        try:
            from getpass import getpass
            token = getpass('HF_TOKEN: ').strip()
        except Exception as exc:
            raise RuntimeError(
                'HF_TOKEN is required. On Kaggle, add a notebook Secret named HF_TOKEN '
                'or HUGGINGFACE_HUB_TOKEN and enable access for this notebook. '
                'Locally, set the HF_TOKEN environment variable before running.'
            ) from exc
    if not token:
        raise RuntimeError('HF_TOKEN is required and was empty.')
    os.environ['HF_TOKEN'] = token
    os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', token)


_resolve_hf_token()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
!pip install -q torch huggingface_hub matplotlib

In [ ]:
import os

defaults = {
    'CANDIDATES_CSV': '',
    'v19_SFT_DATA_REMOTE_PATH': 'data/20260517_074915/candidates_v7.csv',
    'v19_SFT_EPOCHS': '220',
    'v19_SFT_BATCH_GROUPS': '256',
    'v19_SFT_LR': '0.00055',
    'v19_SFT_WEIGHT_DECAY': '0.00028',
    'v19_SFT_DROPOUT': '0.12',
    'v19_SFT_BCE_WEIGHT': '0.24',
    'v19_SFT_PAIR_WEIGHT': '0.30',
    'v19_SFT_RANK_WEIGHT': '0.55',
    'v19_SFT_ENSEMBLE_SIZE': '4',
    'v19_SFT_PATIENCE': '34',
    'v19_SFT_CHECKPOINT_EVERY': '30',
    'v19_SFT_MULTI_GPU': '1',
    'v19_DEVICE': 'cuda',
}
for key, value in defaults.items():
    os.environ.setdefault(key, value)

print('SFT config:', {key: os.environ[key] for key in defaults})

In [ ]:
v19_SFT_CODE = 'import argparse\nimport csv\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\n\nHF_REPO_ID = "devaanshpa/orbit-wars-agent"\nHF_REPO_TYPE = "model"\nHF_REMOTE_PREFIX = "v19/sft"\nPINNED_DATASET = "data/20260517_074915/candidates_v7.csv"\n\nMETADATA_COLS = {\n    "label",\n    "selected",\n    "outcome_weight",\n    "game_result",\n    "reward_margin",\n    "agent_reward",\n    "opponent_reward",\n    "selected_heuristic_rank",\n    "counterfactual_positive",\n    "counterfactual_reason",\n    "failure_overcommit",\n    "failure_missed_tactical",\n    "failure_missed_comet",\n    "failure_slow_expansion",\n    "turn_advantage",\n    "future_advantage_delta_5",\n    "future_advantage_delta_15",\n    "future_advantage_delta_30",\n    "future_production_delta_15",\n    "future_planet_delta_15",\n    "game_id",\n    "candidate_id",\n    "version",\n}\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(\n        description="Train the v19 SFT candidate policy from Orbit Wars candidate groups."\n    )\n    parser.add_argument("--csv", default=os.environ.get("CANDIDATES_CSV", ""))\n    parser.add_argument(\n        "--data-remote-path",\n        default=os.environ.get("v19_SFT_DATA_REMOTE_PATH", PINNED_DATASET),\n        help="Optional exact Hugging Face repo path for candidates_v7.csv. Defaults to the pinned v17/v19 dataset.",\n    )\n    parser.add_argument(\n        "--prefer-local-data",\n        action="store_true",\n        help="Use a local candidates_v7.csv/candidates_v19.csv before trying Hugging Face. Default is Hugging Face.",\n    )\n    parser.add_argument("--export-dir", default="notebooks/v19/exports/sft")\n    parser.add_argument("--epochs", type=int, default=int(os.environ.get("v19_SFT_EPOCHS", "220")))\n    parser.add_argument("--batch-groups", type=int, default=int(os.environ.get("v19_SFT_BATCH_GROUPS", "256")))\n    parser.add_argument("--lr", type=float, default=float(os.environ.get("v19_SFT_LR", "0.00055")))\n    parser.add_argument("--weight-decay", type=float, default=float(os.environ.get("v19_SFT_WEIGHT_DECAY", "0.00028")))\n    parser.add_argument("--dropout", type=float, default=float(os.environ.get("v19_SFT_DROPOUT", "0.12")))\n    parser.add_argument("--bce-weight", type=float, default=float(os.environ.get("v19_SFT_BCE_WEIGHT", "0.24")))\n    parser.add_argument("--pair-weight", type=float, default=float(os.environ.get("v19_SFT_PAIR_WEIGHT", "0.30")))\n    parser.add_argument("--rank-weight", type=float, default=float(os.environ.get("v19_SFT_RANK_WEIGHT", "0.55")))\n    parser.add_argument("--ensemble-size", type=int, default=int(os.environ.get("v19_SFT_ENSEMBLE_SIZE", "4")))\n    parser.add_argument("--patience", type=int, default=int(os.environ.get("v19_SFT_PATIENCE", "34")))\n    parser.add_argument("--checkpoint-every", type=int, default=int(os.environ.get("v19_SFT_CHECKPOINT_EVERY", "30")))\n    parser.add_argument(\n        "--multi-gpu",\n        action="store_true",\n        default=os.environ.get("v19_SFT_MULTI_GPU", "1").strip().lower() not in {"0", "false", "no", "off"},\n        help="Use torch.nn.DataParallel when multiple CUDA devices are available. Enabled by default.",\n    )\n    parser.add_argument(\n        "--device",\n        choices=("cuda", "cpu", "auto"),\n        default=os.environ.get("v19_DEVICE", "cuda"),\n        help="Training device. Defaults to CUDA for Kaggle 2*T4 runs.",\n    )\n    parser.add_argument("--seed", type=int, default=1801)\n    parser.add_argument("--upload", action="store_true")\n    parser.add_argument("--hf-repo-id", default=HF_REPO_ID)\n    parser.add_argument("--hf-repo-type", default=HF_REPO_TYPE)\n    return parser.parse_args()\n\n\ndef load_dotenv(path=".env"):\n    env_path = Path(path)\n    if not env_path.exists():\n        return\n    for raw_line in env_path.read_text(encoding="utf-8").splitlines():\n        line = raw_line.strip()\n        if not line or line.startswith("#") or "=" not in line:\n            continue\n        key, value = line.split("=", 1)\n        key = key.strip()\n        value = value.strip().strip("\\"\'")\n        if key and key not in os.environ:\n            os.environ[key] = value\n\n\ndef download_training_csv(remote_path, repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE):\n    load_dotenv()\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if not token:\n        raise RuntimeError("HF_TOKEN is required to download candidates_v7.csv from Hugging Face.")\n    try:\n        from huggingface_hub import HfApi, hf_hub_download\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to download data: pip install huggingface_hub") from exc\n\n    if not remote_path:\n        api = HfApi(token=token)\n        files = api.list_repo_files(repo_id=repo_id, repo_type=repo_type)\n        remote_csvs = sorted(\n            [\n                name\n                for name in files\n                if name.startswith("data/") and name.endswith("/candidates_v7.csv")\n            ],\n            reverse=True,\n        )\n        if not remote_csvs:\n            raise FileNotFoundError("No data/*/candidates_v7.csv found in Hugging Face repo.")\n        remote_path = remote_csvs[0]\n\n    return Path(\n        hf_hub_download(\n            repo_id=repo_id,\n            repo_type=repo_type,\n            filename=remote_path,\n            token=token,\n        )\n    )\n\n\ndef find_training_csv(csv_arg, remote_path="", prefer_local=False):\n    if csv_arg:\n        path = Path(csv_arg)\n        if not path.exists():\n            raise FileNotFoundError(f"Training CSV does not exist: {path}")\n        return path\n\n    if not prefer_local:\n        return download_training_csv(remote_path)\n\n    local_candidates = []\n    for pattern in (\n        "notebooks/v19/data/candidates_v19.csv",\n        "notebooks/v19/data/candidates_v7.csv",\n    ):\n        path = Path(pattern)\n        if path.exists():\n            return path\n\n    root = Path("data")\n    if root.exists():\n        local_candidates.extend(root.glob("*/candidates_v19.csv"))\n        local_candidates.extend(root.glob("*/candidates_v7.csv"))\n    if local_candidates:\n        return sorted(local_candidates, key=lambda path: path.stat().st_mtime, reverse=True)[0]\n\n    return download_training_csv(remote_path)\n\n\ndef row_float(row, key, default=0.0):\n    try:\n        return float(row.get(key, default) or default)\n    except (TypeError, ValueError):\n        return float(default)\n\n\ndef read_rows(path):\n    with Path(path).open(newline="", encoding="utf-8") as f:\n        rows = list(csv.DictReader(f))\n    if not rows:\n        raise RuntimeError("Training CSV has no rows.")\n    feature_names = [name for name in rows[0] if name not in METADATA_COLS]\n    x_raw = [[row_float(row, name, 0.0) for name in feature_names] for row in rows]\n    labels = [max(0.0, min(1.0, row_float(row, "label", 0.0))) for row in rows]\n    selected = [row_float(row, "selected", 0.0) >= 0.5 for row in rows]\n    counterfactual = [row_float(row, "counterfactual_positive", 0.0) >= 0.5 for row in rows]\n    sample_weights = [max(0.05, row_float(row, "outcome_weight", 1.0)) for row in rows]\n    return rows, feature_names, x_raw, labels, selected, counterfactual, sample_weights\n\n\ndef split_by_game(rows, seed, validation_fraction=0.2):\n    games = sorted({row.get("game_id", "") for row in rows})\n    rng = random.Random(seed)\n    rng.shuffle(games)\n    valid_game_count = max(1, int(len(games) * validation_fraction)) if len(games) > 1 else 1\n    valid_games = set(games[:valid_game_count])\n    valid_indices = [i for i, row in enumerate(rows) if row.get("game_id", "") in valid_games]\n    valid_set = set(valid_indices)\n    train_indices = [i for i in range(len(rows)) if i not in valid_set] or valid_indices[:]\n    return train_indices, valid_indices, games, valid_games\n\n\ndef normalize_from_train(x_raw, train_indices):\n    train_raw = [x_raw[i] for i in train_indices]\n    means = [sum(col) / len(col) for col in zip(*train_raw)]\n    scales = []\n    for j, mean in enumerate(means):\n        var = sum((row[j] - mean) ** 2 for row in train_raw) / max(1, len(train_raw) - 1)\n        scales.append(max(1e-6, math.sqrt(var)))\n\n    def normalize(items):\n        return [[(row[j] - means[j]) / scales[j] for j in range(len(means))] for row in items]\n\n    return means, scales, normalize\n\n\ndef build_groups(rows, indices):\n    grouped = defaultdict(list)\n    for raw_index in indices:\n        step = int(row_float(rows[raw_index], "step", 0.0))\n        grouped[(rows[raw_index].get("game_id", ""), step)].append(raw_index)\n    return [items for items in grouped.values() if len(items) >= 2]\n\n\ndef target_distribution(rows, labels, selected, counterfactual, group):\n    weights = []\n    for raw_index in group:\n        label = labels[raw_index]\n        score = row_float(rows[raw_index], "heuristic_score_scaled", 0.0)\n        turn_delta = (\n            row_float(rows[raw_index], "future_advantage_delta_15", 0.0) * 0.010\n            + row_float(rows[raw_index], "future_production_delta_15", 0.0) * 0.040\n            + row_float(rows[raw_index], "future_planet_delta_15", 0.0) * 0.120\n        )\n        value = max(0.01, label + 0.06 * score + turn_delta)\n        if selected[raw_index]:\n            value += 0.35\n        if counterfactual[raw_index]:\n            value += 0.25\n        if row_float(rows[raw_index], "failure_overcommit", 0.0) >= 0.5:\n            value *= 0.45\n        weights.append(max(0.01, value))\n    total = sum(weights)\n    if total <= 0.0:\n        return [1.0 / len(group)] * len(group)\n    return [weight / total for weight in weights]\n\n\ndef build_hard_pairs(rows, labels, selected, counterfactual, groups, local_index, max_pairs_per_group=8):\n    pairs = []\n    for group in groups:\n        positives = [\n            i\n            for i in group\n            if selected[i] or counterfactual[i] or labels[i] >= 0.55\n        ]\n        negatives = [\n            i\n            for i in group\n            if labels[i] <= 0.35 and not counterfactual[i]\n        ]\n        if not positives or not negatives:\n            continue\n        positives.sort(\n            key=lambda i: labels[i] + row_float(rows[i], "future_advantage_delta_15", 0.0) * 0.004,\n            reverse=True,\n        )\n        negatives.sort(key=lambda i: row_float(rows[i], "heuristic_score_scaled", 0.0), reverse=True)\n        emitted = 0\n        for pos in positives:\n            for neg in negatives:\n                if emitted >= max_pairs_per_group:\n                    break\n                if labels[pos] <= labels[neg] + 0.10:\n                    continue\n                gap = max(0.05, labels[pos] - labels[neg])\n                margin_weight = 1.0 + min(1.5, abs(row_float(rows[pos], "reward_margin", 0.0)) / 650.0)\n                pairs.append((local_index[pos], local_index[neg], gap * margin_weight))\n                emitted += 1\n    return pairs\n\n\ndef sigmoid_prob(value):\n    value = max(-50.0, min(50.0, value))\n    return 1.0 / (1.0 + math.exp(-value))\n\n\ndef grouped_metrics(rows, predictions, positive_mask, indices):\n    groups = build_groups(rows, indices)\n    hits = 0\n    total = 0\n    rank_fractions = []\n    for group in groups:\n        positives = [i for i in group if positive_mask[i]]\n        if not positives:\n            continue\n        ordered = sorted(group, key=lambda i: predictions[i], reverse=True)\n        total += 1\n        if positive_mask[ordered[0]]:\n            hits += 1\n        best_rank = min(ordered.index(i) + 1 for i in positives)\n        rank_fractions.append(best_rank / max(1, len(ordered)))\n    return {\n        "top1": hits / total if total else 0.0,\n        "rank_fraction": sum(rank_fractions) / len(rank_fractions) if rank_fractions else 1.0,\n        "turns": total,\n    }\n\n\ndef tune_blend(rows, probabilities, positive_mask, indices, score_scale):\n    best = {"blend": 0.0, "top1": -1.0, "rank": 1.0}\n    heuristic_scores = [row_float(row, "heuristic_score_scaled", 0.0) * 100.0 for row in rows]\n    model_scores = [(prob - 0.5) * score_scale for prob in probabilities]\n    for step in range(0, 61):\n        blend = step / 100.0\n        mixed = [\n            heuristic_scores[i] * (1.0 - blend) + (heuristic_scores[i] + model_scores[i]) * blend\n            for i in range(len(rows))\n        ]\n        metrics = grouped_metrics(rows, mixed, positive_mask, indices)\n        if metrics["top1"] > best["top1"] or (\n            abs(metrics["top1"] - best["top1"]) <= 1e-9 and metrics["rank_fraction"] < best["rank"]\n        ):\n            best = {"blend": blend, "top1": metrics["top1"], "rank": metrics["rank_fraction"]}\n    return best\n\n\ndef make_model_class(torch, nn):\n    class CandidatePolicy(nn.Module):\n        def __init__(self, feature_count, dropout):\n            super().__init__()\n            self.net = nn.Sequential(\n                nn.Linear(feature_count, 128),\n                nn.ReLU(),\n                nn.Dropout(dropout),\n                nn.Linear(128, 64),\n                nn.ReLU(),\n                nn.Dropout(dropout),\n                nn.Linear(64, 32),\n                nn.ReLU(),\n                nn.Linear(32, 1),\n            )\n\n        def forward(self, inputs):\n            return self.net(inputs).view(-1)\n\n    return CandidatePolicy\n\n\ndef choose_device(torch, args):\n    requested = args.device\n    if requested == "auto":\n        requested = "cuda" if torch.cuda.is_available() else "cpu"\n    if requested == "cuda" and torch.cuda.is_available():\n        return torch.device("cuda"), None\n    if requested == "cuda":\n        print("CUDA requested but unavailable; falling back to CPU.", flush=True)\n    return torch.device("cpu"), None\n\n\ndef optimizer_step(optimizer, xm):\n    if xm is None:\n        optimizer.step()\n    else:\n        xm.optimizer_step(optimizer, barrier=False)\n        xm.mark_step()\n\n\ndef layers_from_model(model, nn):\n    base_model = model.module if hasattr(model, "module") else model\n    linear_layers = [module for module in base_model.net if isinstance(module, nn.Linear)]\n    layers = []\n    for index, layer in enumerate(linear_layers):\n        layers.append(\n            {\n                "weights": layer.weight.detach().cpu().tolist(),\n                "bias": layer.bias.detach().cpu().tolist(),\n                "activation": "relu" if index < len(linear_layers) - 1 else "linear",\n            }\n        )\n    return layers\n\n\ndef maybe_upload_file(args, path, path_in_repo, commit_message):\n    if not args.upload:\n        return\n    load_dotenv()\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if not token:\n        raise RuntimeError("HF_TOKEN is required for checkpoint upload.")\n    try:\n        from huggingface_hub import HfApi\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to upload checkpoints: pip install huggingface_hub") from exc\n    api = HfApi(token=token)\n    api.create_repo(repo_id=args.hf_repo_id, repo_type=args.hf_repo_type, exist_ok=True)\n    api.upload_file(\n        path_or_fileobj=str(path),\n        path_in_repo=path_in_repo,\n        repo_id=args.hf_repo_id,\n        repo_type=args.hf_repo_type,\n        commit_message=commit_message,\n    )\n    print(f"Uploaded checkpoint to https://huggingface.co/{args.hf_repo_id}/blob/main/{path_in_repo}", flush=True)\n\n\ndef save_sft_checkpoint(args, model, nn, member_seed, epoch, item, history, feature_names, means, scales):\n    checkpoint_dir = Path(args.export_dir) / "checkpoints"\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    payload = {\n        "version": "v19_sft_checkpoint",\n        "created_at": int(time.time()),\n        "seed": member_seed,\n        "epoch": epoch,\n        "latest_metrics": item,\n        "history": history,\n        "member": {\n            "version": "v19_sft",\n            "model_type": "mlp_relu_candidate_ranker",\n            "features": feature_names,\n            "mean": dict(zip(feature_names, means)),\n            "scale": dict(zip(feature_names, scales)),\n            "layers": layers_from_model(model, nn),\n            "activation": "relu",\n            "score_scale": 220.0,\n        },\n    }\n    path = checkpoint_dir / f"sft_seed_{member_seed}_epoch_{epoch:03d}.json"\n    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")\n    print(f"Saved v19 SFT checkpoint: {path}", flush=True)\n    maybe_upload_file(\n        args,\n        path,\n        f"{HF_REMOTE_PREFIX}/checkpoints/{path.name}",\n        f"Upload v19 SFT checkpoint seed {member_seed} epoch {epoch}",\n    )\n\n\ndef train_member(torch, nn, functional, args, member_seed, context):\n    (\n        rows,\n        feature_names,\n        means,\n        scales,\n        feature_count,\n        all_x,\n        labels,\n        selected,\n        counterfactual,\n        sample_weights,\n        train_indices,\n        valid_indices,\n        train_groups,\n        valid_groups,\n        train_pairs,\n        valid_pairs,\n        device,\n        xm,\n    ) = context\n    torch.manual_seed(member_seed)\n    random.seed(member_seed)\n    CandidatePolicy = make_model_class(torch, nn)\n    model = CandidatePolicy(feature_count, args.dropout).to(device)\n    if (\n        getattr(args, "multi_gpu", False)\n        and device.type == "cuda"\n        and torch.cuda.device_count() > 1\n    ):\n        print(f"sft seed={member_seed} using DataParallel across {torch.cuda.device_count()} GPUs", flush=True)\n        model = nn.DataParallel(model)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, args.epochs))\n    x_all = torch.tensor(all_x, dtype=torch.float32, device=device)\n    label_tensor = torch.tensor(labels, dtype=torch.float32, device=device)\n    weight_tensor = torch.tensor(sample_weights, dtype=torch.float32, device=device)\n    raw_to_local = {raw: raw for raw in range(len(rows))}\n    best_state = None\n    best_objective = float("inf")\n    stale = 0\n    history = []\n\n    def eval_groups(groups, pairs):\n        model.eval()\n        total_ce = 0.0\n        total_bce = 0.0\n        total_groups = 0\n        with torch.no_grad():\n            logits_all = model(x_all)\n            probs_all = torch.sigmoid(logits_all)\n            for group in groups:\n                indices = torch.tensor(group, dtype=torch.long, device=device)\n                logits = logits_all[indices]\n                targets = torch.tensor(\n                    target_distribution(rows, labels, selected, counterfactual, group),\n                    dtype=torch.float32,\n                    device=device,\n                )\n                total_ce += float(-(targets * functional.log_softmax(logits, dim=0)).sum().detach().cpu())\n                group_labels = label_tensor[indices]\n                group_weights = weight_tensor[indices]\n                total_bce += float(\n                    functional.binary_cross_entropy(probs_all[indices], group_labels, weight=group_weights).detach().cpu()\n                )\n                total_groups += 1\n            pair_loss_value = 0.0\n            if pairs:\n                pos_idx = torch.tensor([p[0] for p in pairs], dtype=torch.long, device=device)\n                neg_idx = torch.tensor([p[1] for p in pairs], dtype=torch.long, device=device)\n                pair_w = torch.tensor([p[2] for p in pairs], dtype=torch.float32, device=device)\n                pair_loss_value = float(\n                    (functional.softplus(-(logits_all[pos_idx] - logits_all[neg_idx])) * pair_w).mean().detach().cpu()\n                )\n            predictions = [sigmoid_prob(value) for value in logits_all.detach().cpu().tolist()]\n        ce = total_ce / max(1, total_groups)\n        bce = total_bce / max(1, total_groups)\n        metrics = grouped_metrics(rows, predictions, [selected[i] or counterfactual[i] or labels[i] >= 0.55 for i in range(len(rows))], valid_indices)\n        objective = ce + args.bce_weight * bce + args.pair_weight * pair_loss_value + args.rank_weight * (1.0 - metrics["top1"])\n        return ce, bce, pair_loss_value, objective, predictions\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        groups = train_groups[:]\n        random.shuffle(groups)\n        total_loss = 0.0\n        batches = 0\n        for offset in range(0, len(groups), args.batch_groups):\n            batch_groups = groups[offset : offset + args.batch_groups]\n            optimizer.zero_grad(set_to_none=True)\n            batch_loss = None\n            flat_group_indices = [raw_index for group in batch_groups for raw_index in group]\n            if flat_group_indices:\n                flat_indices = torch.tensor(flat_group_indices, dtype=torch.long, device=device)\n                flat_logits = model(x_all[flat_indices])\n            cursor = 0\n            for group in batch_groups:\n                width = len(group)\n                if width <= 0:\n                    continue\n                indices = torch.tensor(group, dtype=torch.long, device=device)\n                logits = flat_logits[cursor : cursor + width]\n                cursor += width\n                targets = torch.tensor(\n                    target_distribution(rows, labels, selected, counterfactual, group),\n                    dtype=torch.float32,\n                    device=device,\n                )\n                listwise_loss = -(targets * functional.log_softmax(logits, dim=0)).sum()\n                row_labels = label_tensor[indices]\n                row_weights = weight_tensor[indices]\n                bce_loss = functional.binary_cross_entropy_with_logits(logits, row_labels, weight=row_weights)\n                loss = listwise_loss + args.bce_weight * bce_loss\n                batch_loss = loss if batch_loss is None else batch_loss + loss\n            if batch_loss is None:\n                continue\n            batch_loss = batch_loss / max(1, len(batch_groups))\n            if train_pairs:\n                pair_sample = random.sample(train_pairs, min(len(train_pairs), max(64, args.batch_groups * 4)))\n                pos_idx = torch.tensor([p[0] for p in pair_sample], dtype=torch.long, device=device)\n                neg_idx = torch.tensor([p[1] for p in pair_sample], dtype=torch.long, device=device)\n                pair_w = torch.tensor([p[2] for p in pair_sample], dtype=torch.float32, device=device)\n                pos_logits = model(x_all[pos_idx])\n                neg_logits = model(x_all[neg_idx])\n                pair_loss = (functional.softplus(-(pos_logits - neg_logits)) * pair_w).mean()\n                batch_loss = batch_loss + args.pair_weight * pair_loss\n            batch_loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer_step(optimizer, xm)\n            total_loss += float(batch_loss.detach().cpu())\n            batches += 1\n        scheduler.step()\n        if epoch >= 1:\n            valid_ce, valid_bce, valid_pair, valid_objective, valid_preds = eval_groups(valid_groups, valid_pairs)\n            train_metrics = grouped_metrics(\n                rows,\n                valid_preds,\n                [selected[i] or counterfactual[i] or labels[i] >= 0.55 for i in range(len(rows))],\n                train_indices,\n            )\n            valid_metrics = grouped_metrics(\n                rows,\n                valid_preds,\n                [selected[i] or counterfactual[i] or labels[i] >= 0.55 for i in range(len(rows))],\n                valid_indices,\n            )\n            item = {\n                "epoch": epoch,\n                "train_loss": total_loss / max(1, batches),\n                "valid_ce": valid_ce,\n                "valid_bce": valid_bce,\n                "valid_pair_loss": valid_pair,\n                "valid_objective": valid_objective,\n                "train_turn_top1": train_metrics["top1"],\n                "valid_turn_top1": valid_metrics["top1"],\n                "valid_rank_fraction": valid_metrics["rank_fraction"],\n                "lr": scheduler.get_last_lr()[0],\n            }\n            history.append(item)\n            print(\n                f"sft seed={member_seed} epoch={epoch:03d} "\n                f"loss={item[\'train_loss\']:.5f} valid_obj={valid_objective:.5f} "\n                f"valid_top1={valid_metrics[\'top1\']:.4f} rank={valid_metrics[\'rank_fraction\']:.4f} "\n                f"lr={item[\'lr\']:.6f}",\n                flush=True,\n            )\n            if args.checkpoint_every > 0 and epoch % args.checkpoint_every == 0:\n                save_sft_checkpoint(args, model, nn, member_seed, epoch, item, history, feature_names, means, scales)\n            if valid_objective + 1e-5 < best_objective:\n                best_objective = valid_objective\n                best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n                stale = 0\n            else:\n                stale += 1\n                if stale >= args.patience:\n                    print(f"sft seed={member_seed} early_stop epoch={epoch} best={best_objective:.5f}", flush=True)\n                    break\n\n    if best_state is not None:\n        model.load_state_dict(best_state)\n    model.eval()\n    return model, layers_from_model(model, nn), history, best_objective\n\n\ndef train(args):\n    try:\n        import torch\n        from torch import nn\n        import torch.nn.functional as functional\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("PyTorch is required for v19 SFT training. Install torch or use Kaggle/Colab.") from exc\n\n    random.seed(args.seed)\n    data_path = find_training_csv(\n        args.csv,\n        remote_path=args.data_remote_path,\n        prefer_local=args.prefer_local_data,\n    )\n    rows, feature_names, x_raw, labels, selected, counterfactual, sample_weights = read_rows(data_path)\n    train_indices, valid_indices, games, valid_games = split_by_game(rows, args.seed)\n    means, scales, normalize = normalize_from_train(x_raw, train_indices)\n    all_x = normalize(x_raw)\n    train_groups = build_groups(rows, train_indices)\n    valid_groups = build_groups(rows, valid_indices)\n    train_local = {raw_index: raw_index for raw_index in train_indices}\n    valid_local = {raw_index: raw_index for raw_index in valid_indices}\n    train_pairs = build_hard_pairs(rows, labels, selected, counterfactual, train_groups, train_local)\n    valid_pairs = build_hard_pairs(rows, labels, selected, counterfactual, valid_groups, valid_local)\n    positive_mask = [selected[i] or counterfactual[i] or labels[i] >= 0.55 for i in range(len(rows))]\n    device, xm = choose_device(torch, args)\n    cuda_devices = torch.cuda.device_count() if torch.cuda.is_available() else 0\n\n    print(\n        json.dumps(\n            {\n                "csv": str(data_path),\n                "data_remote_path": args.data_remote_path or "latest data/*/candidates_v7.csv",\n                "data_source": "local_override" if args.csv else ("local_preferred" if args.prefer_local_data else "hugging_face"),\n                "rows": len(rows),\n                "features": len(feature_names),\n                "games": len(games),\n                "validation_games": len(valid_games),\n                "train_groups": len(train_groups),\n                "validation_groups": len(valid_groups),\n                "train_pairs": len(train_pairs),\n                "validation_pairs": len(valid_pairs),\n                "selected_rate": sum(1 for value in selected if value) / len(selected),\n                "counterfactual_rate": sum(1 for value in counterfactual if value) / len(counterfactual),\n                "device": str(device),\n                "cuda_device_count": cuda_devices,\n                "multi_gpu_data_parallel": bool(getattr(args, "multi_gpu", False) and device.type == "cuda" and cuda_devices > 1),\n                "ensemble_size": args.ensemble_size,\n                "checkpoint_every": args.checkpoint_every,\n            },\n            indent=2,\n        ),\n        flush=True,\n    )\n\n    context = (\n        rows,\n        feature_names,\n        means,\n        scales,\n        len(feature_names),\n        all_x,\n        labels,\n        selected,\n        counterfactual,\n        sample_weights,\n        train_indices,\n        valid_indices,\n        train_groups,\n        valid_groups,\n        train_pairs,\n        valid_pairs,\n        device,\n        xm,\n    )\n    members = []\n    histories = []\n    models = []\n    for member_index in range(args.ensemble_size):\n        member_seed = args.seed + member_index * 1009\n        model, layers, history, best_objective = train_member(\n            torch,\n            nn,\n            functional,\n            args,\n            member_seed,\n            context,\n        )\n        models.append(model)\n        histories.append({"seed": member_seed, "history": history, "best_validation_objective": best_objective})\n        members.append(\n            {\n                "version": "v19_sft",\n                "model_type": "mlp_relu_candidate_ranker",\n                "features": feature_names,\n                "mean": dict(zip(feature_names, means)),\n                "scale": dict(zip(feature_names, scales)),\n                "layers": layers,\n                "activation": "relu",\n                "score_scale": 220.0,\n            }\n        )\n\n    with torch.no_grad():\n        all_tensor = torch.tensor(all_x, dtype=torch.float32, device=device)\n        member_probs = []\n        for model in models:\n            logits = model(all_tensor).view(-1).detach().cpu().tolist()\n            member_probs.append([sigmoid_prob(value) for value in logits])\n    all_probs = [sum(member[i] for member in member_probs) / len(member_probs) for i in range(len(rows))]\n    train_metrics = grouped_metrics(rows, all_probs, positive_mask, train_indices)\n    valid_metrics = grouped_metrics(rows, all_probs, positive_mask, valid_indices)\n    tuned_blend = tune_blend(rows, all_probs, positive_mask, valid_indices, 220.0)\n\n    metrics = {\n        "rows": len(rows),\n        "features": len(feature_names),\n        "games": len(games),\n        "train_groups": len(train_groups),\n        "validation_groups": len(valid_groups),\n        "train_pairs": len(train_pairs),\n        "validation_pairs": len(valid_pairs),\n        "selected_rate": sum(1 for value in selected if value) / len(selected),\n        "counterfactual_rate": sum(1 for value in counterfactual if value) / len(counterfactual),\n        "train_turn_top1_positive_rate": train_metrics["top1"],\n        "validation_turn_top1_positive_rate": valid_metrics["top1"],\n        "train_positive_mean_rank_fraction": train_metrics["rank_fraction"],\n        "validation_positive_mean_rank_fraction": valid_metrics["rank_fraction"],\n        "tuned_blend": tuned_blend["blend"],\n        "tuned_blend_validation_top1": tuned_blend["top1"],\n        "tuned_blend_validation_rank_fraction": tuned_blend["rank"],\n        "ensemble_size": len(members),\n        "device": str(device),\n        "cuda_device_count": cuda_devices,\n        "multi_gpu_data_parallel": bool(getattr(args, "multi_gpu", False) and device.type == "cuda" and cuda_devices > 1),\n    }\n    artifact = {\n        "version": "v19_sft",\n        "created_at": int(time.time()),\n        "source_csv": str(data_path),\n        "model_type": "ensemble_mlp_relu_candidate_ranker",\n        "members": members,\n        "blend": tuned_blend["blend"],\n        "metrics": metrics,\n    }\n\n    export_dir = Path(args.export_dir)\n    graph_dir = export_dir / "graphs"\n    export_dir.mkdir(parents=True, exist_ok=True)\n    graph_dir.mkdir(parents=True, exist_ok=True)\n    (export_dir / "model_weights_v19_sft.json").write_text(json.dumps(artifact, indent=2, sort_keys=True), encoding="utf-8")\n    (export_dir / "feature_schema_v19_sft.json").write_text(json.dumps({"features": feature_names}, indent=2), encoding="utf-8")\n    (export_dir / "metrics_v19_sft.json").write_text(json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8")\n    (export_dir / "training_history_v19_sft.json").write_text(json.dumps(histories, indent=2, sort_keys=True), encoding="utf-8")\n    with (export_dir / "predictions_v19_sft.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.writer(f)\n        writer.writerow(["row_index", "label", "prediction", "selected", "counterfactual_positive", "game_result", "split"])\n        valid_set = set(valid_indices)\n        for i, pred in enumerate(all_probs):\n            writer.writerow([\n                i,\n                labels[i],\n                pred,\n                float(selected[i]),\n                float(counterfactual[i]),\n                row_float(rows[i], "game_result", 0.0),\n                "validation" if i in valid_set else "train",\n            ])\n\n    try:\n        import matplotlib.pyplot as plt\n\n        first_history = histories[0]["history"] if histories else []\n        epochs = [item["epoch"] for item in first_history]\n        plt.figure(figsize=(7, 4))\n        plt.plot(epochs, [item["valid_objective"] for item in first_history], label="validation objective")\n        plt.plot(epochs, [item["valid_turn_top1"] for item in first_history], label="validation top1")\n        plt.xlabel("epoch")\n        plt.title("v19 SFT validation")\n        plt.legend()\n        plt.tight_layout()\n        plt.savefig(graph_dir / "sft_validation_v19.png", dpi=150)\n        plt.close()\n\n        plt.figure(figsize=(7, 4))\n        plt.hist([pred for pred, ok in zip(all_probs, positive_mask) if ok], bins=30, alpha=0.65, label="positive")\n        plt.hist([pred for pred, ok in zip(all_probs, positive_mask) if not ok], bins=30, alpha=0.65, label="other")\n        plt.xlabel("policy probability")\n        plt.ylabel("rows")\n        plt.title("v19 SFT prediction distribution")\n        plt.legend()\n        plt.tight_layout()\n        plt.savefig(graph_dir / "sft_prediction_histogram_v19.png", dpi=150)\n        plt.close()\n    except ModuleNotFoundError:\n        print("matplotlib is not installed; skipped graph generation.", flush=True)\n\n    print(json.dumps(metrics, indent=2, sort_keys=True), flush=True)\n    print(f"Saved v19 SFT artifact: {export_dir / \'model_weights_v19_sft.json\'}", flush=True)\n\n    if args.upload:\n        load_dotenv()\n        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n        if not token:\n            raise RuntimeError("HF_TOKEN is required for --upload.")\n        try:\n            from huggingface_hub import HfApi\n        except ModuleNotFoundError as exc:\n            raise RuntimeError("Install huggingface_hub to upload: pip install huggingface_hub") from exc\n        api = HfApi(token=token)\n        api.create_repo(repo_id=args.hf_repo_id, repo_type=args.hf_repo_type, exist_ok=True)\n        api.upload_folder(\n            folder_path=str(export_dir),\n            repo_id=args.hf_repo_id,\n            repo_type=args.hf_repo_type,\n            path_in_repo=HF_REMOTE_PREFIX,\n            commit_message="Upload v19 SFT Orbit Wars policy artifacts and graphs",\n        )\n        print(f"Uploaded {export_dir} to https://huggingface.co/{args.hf_repo_id}/tree/main/{HF_REMOTE_PREFIX}", flush=True)\n\n\ndef main():\n    train(parse_args())\n\n\nif __name__ == "__main__":\n    main()\n'
print(f'Loaded v19 SFT trainer: {len(v19_SFT_CODE)} chars')

In [ ]:
import argparse
import os
from pathlib import Path

export_dir = Path('/kaggle/working/v19_exports/sft') if Path('/kaggle/working').exists() else Path('notebooks/v19/exports/sft')
args = argparse.Namespace(
    csv=os.environ.get('CANDIDATES_CSV', ''),
    data_remote_path=os.environ.get('v19_SFT_DATA_REMOTE_PATH', 'data/20260517_074915/candidates_v7.csv'),
    prefer_local_data=False,
    export_dir=str(export_dir),
    epochs=int(os.environ['v19_SFT_EPOCHS']),
    batch_groups=int(os.environ['v19_SFT_BATCH_GROUPS']),
    lr=float(os.environ['v19_SFT_LR']),
    weight_decay=float(os.environ['v19_SFT_WEIGHT_DECAY']),
    dropout=float(os.environ['v19_SFT_DROPOUT']),
    bce_weight=float(os.environ['v19_SFT_BCE_WEIGHT']),
    pair_weight=float(os.environ['v19_SFT_PAIR_WEIGHT']),
    rank_weight=float(os.environ['v19_SFT_RANK_WEIGHT']),
    ensemble_size=int(os.environ['v19_SFT_ENSEMBLE_SIZE']),
    patience=int(os.environ['v19_SFT_PATIENCE']),
    checkpoint_every=int(os.environ['v19_SFT_CHECKPOINT_EVERY']),
    multi_gpu=os.environ.get('v19_SFT_MULTI_GPU', '1').strip().lower() not in {'0', 'false', 'no', 'off'},
    device=os.environ.get('v19_DEVICE', 'cuda'),
    seed=1801,
    upload=True,
    hf_repo_id='devaanshpa/orbit-wars-agent',
    hf_repo_type='model',
)
print('Running v19 SFT with:', vars(args))
v19_sft_ns = {}
exec(v19_SFT_CODE, v19_sft_ns)
v19_sft_ns['train'](args)